# 👁️ QWEN 2.5-7B VL Fine-Tuning for Poker Perception

**Unsloth-Optimized Training for Google Colab T4**

This notebook fine-tunes QWEN 2.5-7B VL on poker screenshots using:
- 4-bit quantization (fits in 16GB VRAM)
- LoRA for parameter-efficient training
- Automatic checkpointing to Google Drive

## Requirements
- COCO-formatted annotations (annotations.coco.json)
- Image directory with poker screenshots
- Runtime Type: GPU (T4)

## 1️⃣ Setup & Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION - Edit these paths to match your Drive setup
# ═══════════════════════════════════════════════════════════════

CONFIG = {
    # Data paths
    "coco_annotations": "/content/drive/MyDrive/poker_ai/data/annotations.coco.json",
    "images_dir": "/content/drive/MyDrive/poker_ai/data/images",
    
    # Model settings
    "base_model": "Qwen/Qwen2.5-VL-7B-Instruct",
    "output_dir": "/content/drive/MyDrive/poker_ai/models/qwen_poker_vl",
    
    # LoRA settings
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    
    # Training settings
    "batch_size": 1,
    "gradient_accumulation": 8,
    "learning_rate": 2e-5,
    "num_epochs": 3,
    "warmup_ratio": 0.1,
    "max_grad_norm": 1.0,
    "checkpoint_steps": 100,
}

print("✅ Configuration loaded")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")

In [ ]:
# Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q transformers accelerate bitsandbytes peft trl
!pip install -q datasets pillow qwen-vl-utils

print("✅ Dependencies installed")

In [ ]:
# Verify GPU
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    raise RuntimeError("GPU required! Enable in Runtime settings.")

## 2️⃣ Load Dataset

In [ ]:
import json
import os
from pathlib import Path

# Load COCO annotations
with open(CONFIG["coco_annotations"], 'r') as f:
    coco_data = json.load(f)

images = {img['id']: img for img in coco_data.get('images', [])}
annotations = coco_data.get('annotations', [])

print(f"Loaded {len(images)} images and {len(annotations)} annotations")

In [ ]:
# Prepare training data
DETECTION_PROMPT = """Analyze this poker game screenshot and extract the exact game state.
Return ONLY a JSON object with:
- hole_cards: List of player's cards (e.g., ["As", "Kh"])
- community_cards: List of board cards
- pot_size: Integer pot amount
- current_bet: Integer bet to call
- player_stack: Player's chip count
- opponent_stack: Opponent's chip count
- street: "preflop", "flop", "turn", or "river"
- action_required: true/false
- available_actions: List of valid actions"""

training_data = []

for ann in annotations:
    image_info = images.get(ann['image_id'], {})
    image_path = os.path.join(CONFIG["images_dir"], image_info.get('file_name', ''))
    
    if not os.path.exists(image_path):
        continue
    
    # Build game state from annotation
    game_state = ann.get('game_state', {})
    response = json.dumps(game_state, indent=2)
    
    training_data.append({
        'image': image_path,
        'prompt': DETECTION_PROMPT,
        'response': response
    })

print(f"Prepared {len(training_data)} training samples")

## 3️⃣ Load Model with Unsloth

In [ ]:
from unsloth import FastVisionModel

# Load model with 4-bit quantization
model, tokenizer = FastVisionModel.from_pretrained(
    CONFIG["base_model"],
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

print(f"\n✅ Model loaded")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Apply LoRA
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print("✅ LoRA applied")

## 4️⃣ Train Model

In [ ]:
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Convert to HuggingFace dataset
dataset = Dataset.from_list(training_data)

# Training configuration
training_args = SFTConfig(
    output_dir=CONFIG["output_dir"],
    per_device_train_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation"],
    learning_rate=CONFIG["learning_rate"],
    num_train_epochs=CONFIG["num_epochs"],
    warmup_ratio=CONFIG["warmup_ratio"],
    max_grad_norm=CONFIG["max_grad_norm"],
    fp16=True,
    logging_steps=10,
    save_steps=CONFIG["checkpoint_steps"],
    save_total_limit=3,
    seed=42,
    report_to="none",
)

print("✅ Training config ready")

In [ ]:
# Initialize trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

# Train!
print("Starting training...")
print("="*50)

trainer.train()

print("\n✅ Training complete!")

## 5️⃣ Save Model

In [ ]:
# Save final model
model.save_pretrained(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"✅ Model saved to {CONFIG['output_dir']}")

In [ ]:
# Also save as merged 16-bit for faster inference
merged_dir = CONFIG["output_dir"] + "_merged"

model.save_pretrained_merged(
    merged_dir,
    tokenizer,
    save_method="merged_16bit",
)

print(f"✅ Merged model saved to {merged_dir}")

## 6️⃣ Test Inference

In [ ]:
from PIL import Image

# Test with a sample image
if len(training_data) > 0:
    test_sample = training_data[0]
    test_image = Image.open(test_sample['image'])
    
    # Prepare messages
    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": DETECTION_PROMPT}
        ]}
    ]
    
    # Apply chat template
    input_text = tokenizer.apply_chat_template(
        messages, 
        add_generation_prompt=True
    )
    
    # Tokenize
    inputs = tokenizer(
        images=test_image,
        text=input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")
    
    # Generate
    FastVisionModel.for_inference(model)
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        use_cache=True,
        temperature=1.0,
    )
    
    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print("Test Inference Result:")
    print("="*50)
    print(response[-500:])  # Last 500 chars
else:
    print("No training data available for testing")

In [ ]:
# Cleanup
import gc

del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()

vram_used = torch.cuda.memory_allocated() / 1e9
print(f"VRAM after cleanup: {vram_used:.2f} GB")
print("\n✅ Training complete! Model saved to Google Drive.")